# **Predictive Model Evaluation**

## Objectives

* Extend notebook [09_predictive_modelling](/jupyter_notebooks/09_predictive_modelling.ipynb) to evaluate the fitted classification pipeline's performance
* Assess feature importance to understand model behaviour
* Revisit modelling choices and refit if performance does not meet requirements
* Answer Business Requirement 2: *TCS Hotels wants a machine learning model capable of predicting the likelihood of a booking cancellation, accessed through an operational dashboard that supports the reservations team in three ways: a risk report of upcoming arrivals, individual reservation search and a prospective booking risk assessor*

## Inputs

* Classification Pipeline "outputs/ml_pipeline/cancel_predict/v1/classification_model_pipeline.pkl"
* Train and test datasets from "outputs/ml_pipeline/preprocessing"

## Outputs

* Confusion matrix and classification report for train and test sets
* Feature importance plot
* Updated preprocessing pipeline saved to "outputs/ml_pipeline/cancel_predict/v2/classification_preprocessing_pipeline.pkl"
* Updated modelling pipeline saved to "outputs/ml_pipeline/cancel_predict/v2/classification_model_pipeline.pkl"
* Trained classification model for Business Requirement 2, to be integrated into the operational dashboard

## Additional Comments

* Feature importance analysis identified a potential data leakage risk in `deposit_type`. This was investigated and validated through an ablation study, resulting in its removal from the final (v2) pipeline. See the Feature Importance section for full reasoning.


---

# Change working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chdir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))

current_dir = os.getcwd()
current_dir

---

## Load Data

In [ ]:
import pandas as pd
import joblib

X_train = pd.read_csv("outputs/ml_pipeline/preprocessing/X_train.csv")
X_test = pd.read_csv("outputs/ml_pipeline/preprocessing/X_test.csv")
y_train = pd.read_csv("outputs/ml_pipeline/preprocessing/y_train.csv").squeeze()
y_test = pd.read_csv("outputs/ml_pipeline/preprocessing/y_test.csv").squeeze()
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)


* Load the fitted pipeline

In [ ]:
classification_model_pipeline = joblib.load(
    "outputs/ml_pipeline/cancel_predict/v1/classification_model_pipeline.pkl"
)
classification_model_pipeline

---

## Generate Predictions

In [ ]:
y_train_pred = classification_model_pipeline.predict(X_train)
y_test_pred = classification_model_pipeline.predict(X_test)

---

## Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="winter",
                xticklabels=["Not Cancelled", "Cancelled"],
                yticklabels=["Not Cancelled", "Cancelled"])
    plt.title(title)
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.show()


In [ ]:
plot_confusion_matrix(y_train, y_train_pred, "Train Set")
plot_confusion_matrix(y_test, y_test_pred, "Test Set")

* It appears as though there are more errors generated on the test set suggesting some overfitting

---

## Classification Report

In [ ]:
from sklearn.metrics import classification_report

print("TRAIN SET\n")
print(classification_report(y_train, y_train_pred, target_names=["Not Cancelled", "Cancelled"]))

print("TEST SET\n")
print(classification_report(y_test, y_test_pred, target_names=["Not Cancelled", "Cancelled"]))

* This test confirms overfitting of the model, displaying near perfect scores on all metrics on the train set.
* The model is still hitting the target recall value of 0.8, but given the .15 difference in recall between the 2 sets, revisiting the hyperparameters is appropriate.

---

## Additional Tuning to Reduce Overfitting

* Copy over the unfit pipeline and hyperparameter function from the [predictive modelling](/jupyter_notebooks/09_predictive_modelling.ipynb) notebook

In [ ]:
classification_model_preprocessing_pipeline = joblib.load("outputs/ml_pipeline/preprocessing/classification_preprocessing_pipeline.pkl")

In [ ]:
from sklearn.pipeline import Pipeline
from utils.custom_transformers import undefined_meal

def classification_pipeline(model):
    pipeline_base = Pipeline([
        ("Preprocessing", classification_model_preprocessing_pipeline),
        ("model", model)
    ])

    return pipeline_base

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import cross_validate

def parameter_comparison(model, param, values):

    results = []

    for value in values:

        current_model = clone(model)
        current_model.set_params(**{param: value})

        scores = cross_validate(
            classification_pipeline(current_model),
            X_train,
            y_train.values.ravel(),
            scoring="recall",
            cv=5,
            return_train_score=True,
            verbose=1
        )

        results.append({
            "parameter": param,
            "value": value,
            "train_recall": scores["train_score"].mean(),
            "val_recall": scores["test_score"].mean(),
            "diff": scores["train_score"].mean() - scores["test_score"].mean()
        })

    return pd.DataFrame(results)

* I suspect that the increased tree depth is largely responsible for the overfitting, I will keep the carried forward n_estimators and retest max_depth to retrieve more information

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(random_state=0, n_estimators=500)

max_depth_results = parameter_comparison(
    model=model,
    param="max_depth",
    values=[6, 8, 10]
)

max_depth_results

* This shows that the smallest gap between training and validation is when max_depthe is set to the default value of 6
* Now to also re-test the n_estimators values using the default max_depth value

In [ ]:
model = XGBClassifier(random_state=0)

n_estimators_results = parameter_comparison(
    model=model,
    param="n_estimators",
    values=[100, 200, 500]
)

n_estimators_results

* From this test we can see that the gap between variance increases with estimators. Rather than reducing estimators, I will try other parameters that help to reduce overfitting to see if there is improvement to be had there.

In [ ]:
model = XGBClassifier(random_state=0, n_estimators=500)

min_child_weight_results = parameter_comparison(
    model=model,
    param="min_child_weight",
    values=[1, 5, 10, 20, 50]
)

min_child_weight_results

* min_child_weight of 10 reduced the variance by 3 points without significantly damaging recall so will be carried forward

In [ ]:
model = XGBClassifier(random_state=0, n_estimators=500, min_child_weight=10)

gamma_results = parameter_comparison(
    model=model,
    param="gamma",
    values=[0, 0.1, 0.5, 1, 5]
)

gamma_results

* The gamma value of 0.1 has reduced variance by a further 2 points without significant impact on the recall score.

### Update the pipeline and repeat classification report with the updated hyperparameters

In [ ]:
def classification_pipeline():
    pipeline_base = Pipeline([
        ("Preprocessing", classification_model_preprocessing_pipeline),
        ("model", XGBClassifier(random_state=0, n_estimators=500, min_child_weight=10, gamma=0.1))
    ])

    return pipeline_base

* Fit to training data

In [ ]:
X = X_train.copy()
y = y_train.copy()

updated_classification_model_pipeline = classification_pipeline()
updated_classification_model_pipeline.fit(X, y)

In [ ]:
y_train_pred = updated_classification_model_pipeline.predict(X_train)
y_test_pred = updated_classification_model_pipeline.predict(X_test)

In [ ]:
print("TRAIN SET\n")
print(classification_report(y_train, y_train_pred, target_names=["Not Cancelled", "Cancelled"]))

print("TEST SET\n")
print(classification_report(y_test, y_test_pred, target_names=["Not Cancelled", "Cancelled"]))

* Repeat confusion matrix

In [ ]:
plot_confusion_matrix(y_train, y_train_pred, "Train Set")
plot_confusion_matrix(y_test, y_test_pred, "Test Set")

In [ ]:
from sklearn.metrics import recall_score

train_recall = recall_score(y_train, y_train_pred, pos_label=1)
test_recall = recall_score(y_test, y_test_pred, pos_label=1)

target_recall = 0.80

print(f"Train recall: {train_recall:.2f}")
print(f"Test recall: {test_recall:.2f}")
print(f"Target recall met on test set: {test_recall >= target_recall}")

* The retuned model reduced the train/test recall gap from 15.6 to 6.0 points, indicating substantially less overfitting, while test recall of 0.82 continues to exceed the 0.80 business target, albeit with a narrower margin than the original model (0.84)

---

## Feature Importance

In [ ]:
preprocessed_sample = classification_model_pipeline.named_steps["Preprocessing"].transform(X_train)
feature_names = preprocessed_sample.columns

model = classification_model_pipeline.named_steps["model"]
importances = model.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

feature_importance_df.head(15)


In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(data=feature_importance_df.head(15), x="Importance", y="Feature")
plt.title("Top 15 Feature Importances — Cancellation Prediction")
plt.show()

* deposit_type_Non_Refund is dominating the feature importance to such an extent as to raise questions bout whether it is a genuine deposit or some other mechanism such as full payment taken at the point of cancellation which could be causing data leakage.

* Test model performace without `deposit_type`

In [ ]:
from feature_engine.selection import DropFeatures
from feature_engine.imputation import ArbitraryNumberImputer, CategoricalImputer
from feature_engine.outliers import Winsorizer
from feature_engine.encoding import OneHotEncoder, RareLabelEncoder, OrdinalEncoder
from sklearn.preprocessing import FunctionTransformer

def classification_pipeline_no_deposit():
    pipeline_base = Pipeline([
        ("DropFeatures", DropFeatures(features_to_drop=["company", "arrival_date_year", "deposit_type", "arrival_date_week_number"])),
        ("FunctionTransformer", FunctionTransformer(undefined_meal)),
        ("ArbitraryNumberImputer", ArbitraryNumberImputer(arbitrary_number=0, variables="agent")),
        ("CategoricalImputer", CategoricalImputer(imputation_method="frequent", variables="country")),
        ("Winsorizer", Winsorizer(capping_method="iqr", tail="right", fold=1.5, variables=["lead_time", "adr",
                                                                                           "stays_in_weekend_nights", "stays_in_week_nights"])),
        ("RareLabelEncoder", RareLabelEncoder(tol=0.01, variables="country")),
        ("OrdinalEncoder", OrdinalEncoder(encoding_method="arbitrary", variables="country", ignore_format=True)),
        ("MonthEncoder", OneHotEncoder(variables="arrival_date_month")) ,
        ("OneHotEncoder", OneHotEncoder(variables=["hotel", "meal", "market_segment", "distribution_channel",
                                                   "reserved_room_type", "assigned_room_type", "customer_type"], drop_last=True)),
        ("model", XGBClassifier(random_state=0, n_estimators=500, min_child_weight=10, gamma=0.1))
    ])

    return pipeline_base

In [ ]:
X_train_no_deposit = X_train.copy()
X_test_no_deposit = X_test.copy()

classification_model_pipeline_no_deposit = classification_pipeline_no_deposit()
classification_model_pipeline_no_deposit.fit(X_train_no_deposit, y)

In [ ]:
y_train_pred_no_deposit = classification_model_pipeline_no_deposit.predict(X_train)
y_test_pred_no_deposit = classification_model_pipeline_no_deposit.predict(X_test)

In [ ]:
print("TRAIN SET\n")
print(classification_report(y_train, y_train_pred_no_deposit, target_names=["Not Cancelled", "Cancelled"]))

print("TEST SET\n")
print(classification_report(y_test, y_test_pred_no_deposit, target_names=["Not Cancelled", "Cancelled"]))

* Feature importance identified deposit_type as the dominant predictor. However, an ablation study in which the feature was removed showed virtually no degradation in model performance, indicating that the predictive information contained in deposit_type is largely captured by other variables.

| Metric (Cancelled class) | With deposit_type | Without deposit_type | Difference |
| --- | --- | --- | --- |
| Precision | 0.86 | 0.85 | -0.01 |
| Recall | 0.82 | 0.82 | 0.00 |
| F1-score | 0.84 | 0.83 | -0.01 |
| Accuracy | 0.88 | 0.88 | 0.00 |


* Run a comparison feature importance

In [ ]:
sample_preprocessed = classification_model_pipeline_no_deposit[:-1].transform(X_train)
feature_names = sample_preprocessed.columns

model = classification_model_pipeline_no_deposit.named_steps["model"]
importances = model.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

feature_importance_df.head(15)

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(data=feature_importance_df.head(15), x="Importance", y="Feature")
plt.title("Top 15 Feature Importances — Cancellation Prediction")
plt.show()

In [ ]:
from sklearn.model_selection import cross_validate

results = {}
pipelines = {
    "with_deposit": updated_classification_model_pipeline,
    "without_deposit": classification_model_pipeline_no_deposit
}

for name, pipeline in pipelines.items():

    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring='f1',
        return_train_score=False,
        n_jobs=-1
    )

    results[name] = {
        "Mean CV F1": cv_results["test_score"].mean(),
        "CV Std F1": cv_results["test_score"].std(),
        "Individual Scores": cv_results["test_score"]
    }


cv_comparison = pd.DataFrame(results).T
cv_comparison

* The F1 score comparison confirms that there is insignificant impact on the results to warrant keeping the deposit_type feature
* There is ambiguity in how the variable is derived leading to concerns that there could be data-leakage: 

>Value calculated based on the payments identified for the booking in the transaction (TR) table before the booking׳s arrival or cancellation date. <br>
>Non Refund – a deposit was made in the value of the total stay cost; [*Hotel booking demand datasets*](https://pmc.ncbi.nlm.nih.gov/articles/PMC6297060/)

* Although it was the most influential feature in the initial model (accounting for 87% of feature importance), its calculation relies on transaction information recorded before the booking’s arrival or cancellation date, raising uncertainty around whether this information would be available at the point of prediction. 
* In addition, the strong association between the Non Refund category and cancellations (Non Refund carries a 99% cancellation rate) suggests that the feature may act as a proxy for the target rather than capturing independent predictive behaviour. 
* Ablation testing showed that removing `deposit_type` resulted in only a marginal reduction in model performance (~1% decrease in F1 score and precision), while producing a more balanced feature importance distribution and a more robust, interpretable model. 
* Therefore, `deposit_type` will be removed from the final model to reduce leakage risk or model over-reliance without materially impacting predictive performance.

* Update the preprocessing pipelines to reflect the removal of `deposit_type`

In [ ]:
outlier_cols = ["lead_time", "adr", "stays_in_weekend_nights", "stays_in_week_nights"]
categorical_cols = ["hotel", "meal", "market_segment", "distribution_channel", "reserved_room_type", "assigned_room_type", "customer_type"]

def preprocessing():

    pipeline_base = Pipeline([
        ("DropFeatures", DropFeatures(features_to_drop=["company", "arrival_date_year", "deposit_type", "arrival_date_week_number"])),
        ("FunctionTransformer", FunctionTransformer(undefined_meal)),
        ("ArbitraryNumberImputer", ArbitraryNumberImputer(arbitrary_number=0, variables="agent")),
        ("CategoricalImputer", CategoricalImputer(imputation_method="frequent", variables="country")),
        ("Winsorizer", Winsorizer(capping_method="iqr", tail="right", fold=1.5, variables=outlier_cols)),
        ("RareLabelEncoder", RareLabelEncoder(tol=0.01, variables="country")),
        ("OrdinalEncoder", OrdinalEncoder(encoding_method="arbitrary", variables="country", ignore_format=True)),
        ("MonthEncoder", OneHotEncoder(variables="arrival_date_month")), 
        ("OneHotEncoder", OneHotEncoder(variables=categorical_cols, drop_last=True))
    ])

    return pipeline_base

preprocessing_pipeline = preprocessing()

In [ ]:
def prediction():

    pipeline_base = Pipeline([
        ("Preprocessing", preprocessing_pipeline),
        ("model", XGBClassifier(random_state=0, n_estimators=500, min_child_weight=10, gamma=0.1))
    ])

    return pipeline_base

prediction_pipeline = prediction()

In [ ]:
X = X_train.copy()
y = y_train.copy()

prediction_pipeline.fit(X, y)

In [ ]:
y_train_pred_final = prediction_pipeline.predict(X_train)
y_test_pred_final = prediction_pipeline.predict(X_test)

print("TRAIN SET\n")
print(classification_report(y_train, y_train_pred_final, target_names=["Not Cancelled", "Cancelled"]))
print("TEST SET\n")
print(classification_report(y_test, y_test_pred_final, target_names=["Not Cancelled", "Cancelled"]))

---

## Conclusions

**Overfitting**
* Initial evaluation of the v1 pipeline showed a substantial train/test recall gap (15.6 point difference) combined with near perfect train set scores (1.00 or 0.99 across all metrics). This suggested the model was overfitting to the training data despite meeting the 0.80 recall target on the test set. 
* Targeted hyperparameter tuning — reducing `max_depth` to its default, increasing `min_child_weight` to 10, and adding `gamma=0.1` — reduced this gap to 6.0 points (train 0.88, test 0.82) with only marginal cost to test recall, producing a model with a more realistic generalisation profile.

**Feature Importance**
* Feature importance analysis of the retuned model showed `deposit_type_Non_Refund` accounting for approximately 87% of total importance, far exceeding any other feature. 
* The deposit type "Non Refund" had a cancellation rate of 99% as discovered in the [cancellation eda](/jupyter_notebooks/02_cancellation_eda.ipynb) notebook and combined with the feature's high importance in the model, this raised a concern that this feature could be acting as a proxy for `is_canceled`rather than an independent predictor
* Investigation of the feature's derivation raised a data leakage concern: it is calculated from transaction records that may only be finalised at or near the point of cancellation, meaning it could encode information not genuinely available at prediction time. 
* An ablation study confirmed the feature was not essential to model performance — removing it produced a negligible drop in F1, precision, and recall (≤0.01 across all three) — while yielding a more balanced feature importance distribution across the remaining predictors. 
* `deposit_type` was therefore removed from the final pipeline (v2) to reduce leakage risk and produce a more interpretable, robust model without a meaningful performance cost.

**Business Requirement 2**
* The final v2 pipeline achieves a test recall of 0.82 for the Cancelled class, exceeding the 0.80 target defined in the ML Business Case. Combined with the reduced overfitting gap and removal of a feature carrying leakage risk, the model is considered fit for integration into the operational dashboard to answer Business Requirement 2.

**Limitations and Next Steps**
* The remaining ~6 point train/test gap suggests some overfitting persists; further regularisation (`reg_alpha`/`reg_lambda`) could be explored in a future iteration if time allows.
* `deposit_type`'s removal was justified via ablation on this dataset only; if the feature's true derivation timing can be confirmed with the data provider, its reinstatement could be reconsidered.

---

## Save Files

In [ ]:
import os
try:
  os.makedirs(name='outputs/ml_pipeline/cancel_predict/v2')
except Exception as e:
  print(e)


* Save updated preprocessing pipeline

In [ ]:
joblib.dump(value=preprocessing_pipeline, filename="outputs/ml_pipeline/cancel_predict/v2/classification_preprocessing_pipeline.pkl")

* Save the updated prediction pipeline

In [ ]:
joblib.dump(value=prediction_pipeline, filename="outputs/ml_pipeline/cancel_predict/v2/classification_model_pipeline.pkl")